# Modul 5: GPU Computing Dasar – CUDA
## Langkah 1: Setup dan Verifikasi Lingkungan



In [1]:
import subprocess, os, re
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Verifikasi NVCC
r = subprocess.run(['nvcc','--version'], capture_output=True, text=True)
print('=== NVCC Version ==='); print(r.stdout.strip())

# Info GPU
r2 = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print('\n=== GPU Info ==='); print(r2.stdout)
def check_command(command):
    """Memeriksa apakah perintah dapat dijalankan di sistem."""
    try:
        r = subprocess.run(command, capture_output=True, text=True, shell=True)
        return r.returncode == 0, r.stdout.strip()
    except Exception:
        return False, ""

print("🔍 Memeriksa Kebutuhan Dasar CUDA Programming...\n")

# 1. Verifikasi NVIDIA Driver via nvidia-smi
has_smi, smi_out = check_command('nvidia-smi')
print('=== 1. GPU & Driver Info (nvidia-smi) ===')
if has_smi:
    print("\n".join(smi_out.split('\n')[:5])) 
else:
    print("❌ Driver NVIDIA tidak terdeteksi atau perintah 'nvidia-smi' tidak ditemukan.")

print('\n' + '='*40 + '\n')

# 2. Verifikasi CUDA Compiler (nvcc)
has_nvcc, nvcc_out = check_command('nvcc --version')
print('=== 2. NVCC Version ===')
if has_nvcc:
    print(nvcc_out)
else:
    print("❌ CUDA Compiler (nvcc) TIDAK ditemukan di PATH.")

print('\n' + '='*40 + '\n')

# 3. Kesimpulan Evaluasi
if has_smi and has_nvcc:
    print("✅ SEMUA AMAN! Environment Anda sudah siap untuk CUDA programming.")
    print("Lompat langsung ke Cell 4 (Setup Direktori).")
else:
    print("⚠️ Environment BELUM siap. Silakan instal cuda-nvcc di conda.")


=== NVCC Version ===
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2026 NVIDIA Corporation
Built on Fri_Apr_24_07:22:02_PM_PDT_2026
Cuda compilation tools, release 13.3, V13.3.33
Build cuda_13.3.r13.3/compiler.37862127_0

=== GPU Info ===
Sun Jun 14 01:11:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.110                Driver Version: 581.86         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...    On  |   00000000:01:00.0  On |      

In [2]:
# Buat direktori kerja
os.makedirs('cuda_files', exist_ok=True)
print('Direktori cuda_files siap.')

# Cek compute capability GPU
r3 = subprocess.run(
    'nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv,noheader',
    shell=True, capture_output=True, text=True)
print('GPU Details:', r3.stdout.strip())

Direktori cuda_files siap.
GPU Details: NVIDIA GeForce RTX 3050 6GB Laptop GPU, 8.6, 6144 MiB


## Langkah 2: CPU Serial – Baseline

In [3]:
%%writefile cuda_files/vector_add_cpu.cpp
#include <iostream>
#include <vector>
#include <chrono>
#include <cstdlib>
#include <cmath>

void vector_add_cpu(float* A, float* B, float* C, int N) {
    for (int i = 0; i < N; i++) C[i] = A[i] + B[i];
}

int main(int argc, char* argv[]) {
    int N = (argc > 1) ? atoi(argv[1]) : 10000000;
    std::vector<float> A(N), B(N), C(N);
    srand(42);
    for (int i = 0; i < N; i++) {
        A[i] = (float)rand()/RAND_MAX;
        B[i] = (float)rand()/RAND_MAX;
    }
    auto t0 = std::chrono::high_resolution_clock::now();
    vector_add_cpu(A.data(), B.data(), C.data(), N);
    auto t1 = std::chrono::high_resolution_clock::now();
    double ms = std::chrono::duration<double,std::milli>(t1-t0).count();
    bool ok = true;
    for (int i=0;i<10;i++) if (fabs(C[i]-(A[i]+B[i]))>1e-6){ok=false;break;}
    float bw = (3.0f*N*sizeof(float))/(ms/1000.0f)/1e9;
    printf("N=%d TIME_MS=%.4f BW=%.4f CORRECT=%d\n",N,ms,bw,(int)ok);
    return 0;
}


Overwriting cuda_files/vector_add_cpu.cpp


In [4]:
!g++ -O2 -std=c++11 cuda_files/vector_add_cpu.cpp -o cuda_files/vec_cpu
!./cuda_files/vec_cpu 10000000   # uji dengan 10 juta elemen


/bin/bash: line 1: g++: command not found
/bin/bash: line 1: ./cuda_files/vec_cpu: No such file or directory


In [5]:
%%writefile cuda_files/vector_add_openmp.cpp
#include <iostream>
#include <vector>
#include <chrono>
#include <cstdlib>
#include <cmath>
#include <omp.h>

int main(int argc, char* argv[]) {
    int N = (argc>1)?atoi(argv[1]):10000000;
    int T = (argc>2)?atoi(argv[2]):4;
    std::vector<float> A(N), B(N), C(N);
    srand(42);
    for (int i=0;i<N;i++){A[i]=(float)rand()/RAND_MAX;B[i]=(float)rand()/RAND_MAX;}
    omp_set_num_threads(T);
    auto t0 = std::chrono::high_resolution_clock::now();
    #pragma omp parallel for
    for (int i=0;i<N;i++) C[i]=A[i]+B[i];
    auto t1 = std::chrono::high_resolution_clock::now();
    double ms = std::chrono::duration<double,std::milli>(t1-t0).count();
    bool ok=true;
    for(int i=0;i<10;i++) if(fabs(C[i]-(A[i]+B[i]))>1e-6){ok=false;break;}
    float bw=(3.0f*N*sizeof(float))/(ms/1000.0f)/1e9;
    printf("N=%d THREADS=%d TIME_MS=%.4f BW=%.4f CORRECT=%d\n",N,T,ms,bw,(int)ok);
    return 0;
}


Overwriting cuda_files/vector_add_openmp.cpp


In [3]:
!g++ -O2 -std=c++11 -fopenmp cuda_files/vector_add_openmp.cpp -o cuda_files/vec_omp
!./cuda_files/vec_omp 10000000 4   # 10M elemen, 4 thread


N=10000000 THREADS=4 TIME_MS=6.5693 BW=18.2669 CORRECT=1


## Langkah 3: CPU dengan OpenMP (Parallel Baseline)

In [5]:
%%writefile cuda_files/vector_add_cuda.cu
#include <iostream>
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

#define CHECK(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA error %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(1); \
    } \
} while (0)

__global__ void vec_add(float* A, float* B, float* C, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        C[idx] = A[idx] + B[idx];
    }
}

int main(int argc, char* argv[]) {
    int N = (argc > 1) ? atoi(argv[1]) : 10000000;
    int TPB = (argc > 2) ? atoi(argv[2]) : 256;
    size_t sz = N * sizeof(float);

    float *h_A = new float[N];
    float *h_B = new float[N];
    float *h_C = new float[N];

    srand(42);
    for (int i = 0; i < N; i++) {
        h_A[i] = (float)rand() / RAND_MAX;
        h_B[i] = (float)rand() / RAND_MAX;
    }

    float *d_A, *d_B, *d_C;
    CHECK(cudaMalloc(&d_A, sz));
    CHECK(cudaMalloc(&d_B, sz));
    CHECK(cudaMalloc(&d_C, sz));

    cudaEvent_t ev0, ev1;
    CHECK(cudaEventCreate(&ev0));
    CHECK(cudaEventCreate(&ev1));

    float t_htod = 0.0f;
    float t_kernel = 0.0f;
    float t_dtoh = 0.0f;

    CHECK(cudaEventRecord(ev0));
    CHECK(cudaMemcpy(d_A, h_A, sz, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_B, h_B, sz, cudaMemcpyHostToDevice));
    CHECK(cudaEventRecord(ev1));
    CHECK(cudaEventSynchronize(ev1));
    CHECK(cudaEventElapsedTime(&t_htod, ev0, ev1));

    int blocks = (N + TPB - 1) / TPB;

    CHECK(cudaEventRecord(ev0));
    vec_add<<<blocks, TPB>>>(d_A, d_B, d_C, N);
    CHECK(cudaGetLastError());
    CHECK(cudaEventRecord(ev1));
    CHECK(cudaEventSynchronize(ev1));
    CHECK(cudaEventElapsedTime(&t_kernel, ev0, ev1));

    CHECK(cudaEventRecord(ev0));
    CHECK(cudaMemcpy(h_C, d_C, sz, cudaMemcpyDeviceToHost));
    CHECK(cudaEventRecord(ev1));
    CHECK(cudaEventSynchronize(ev1));
    CHECK(cudaEventElapsedTime(&t_dtoh, ev0, ev1));

    float total = t_htod + t_kernel + t_dtoh;

    bool ok = true;
    for (int i = 0; i < 10; i++) {
        if (fabs(h_C[i] - (h_A[i] + h_B[i])) > 1e-6) {
            ok = false;
            break;
        }
    }

    printf("N=%d TPB=%d BLOCKS=%d HTOD=%.4f KERNEL=%.4f DTOH=%.4f TOTAL=%.4f CORRECT=%d\n",
           N, TPB, blocks, t_htod, t_kernel, t_dtoh, total, (int)ok);

    delete[] h_A;
    delete[] h_B;
    delete[] h_C;

    CHECK(cudaFree(d_A));
    CHECK(cudaFree(d_B));
    CHECK(cudaFree(d_C));

    CHECK(cudaEventDestroy(ev0));
    CHECK(cudaEventDestroy(ev1));

    return 0;
}

Overwriting cuda_files/vector_add_cuda.cu


In [7]:
!nvcc -O2 -std=c++11 -arch=sm_86 cuda_files/vector_add_cuda.cu -o cuda_files/vec_cuda
!./cuda_files/vec_cuda 10000000 256


N=10000000 TPB=256 BLOCKS=39063 HTOD=17.2556 KERNEL=2.7758 DTOH=48.7082 TOTAL=68.7396 CORRECT=1


## Langkah 4: CUDA Basic – GPU Kernel

In [10]:
%%writefile cuda_files/vector_add_cuda.cu
#include <iostream>
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

void checkCuda(cudaError_t err, const char* file, int line) {
    if (err != cudaSuccess) {
        printf("CUDA error %s:%d: %s\n", file, line, cudaGetErrorString(err));
        exit(1);
    }
}

#define CHECK(call) checkCuda((call), __FILE__, __LINE__)

__global__ void vec_add(float* A, float* B, float* C, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        C[idx] = A[idx] + B[idx];
    }
}

int main(int argc, char* argv[]) {
    int N = (argc > 1) ? atoi(argv[1]) : 10000000;
    int TPB = (argc > 2) ? atoi(argv[2]) : 256;
    size_t sz = N * sizeof(float);

    float *h_A = new float[N];
    float *h_B = new float[N];
    float *h_C = new float[N];

    srand(42);
    for (int i = 0; i < N; i++) {
        h_A[i] = (float)rand() / RAND_MAX;
        h_B[i] = (float)rand() / RAND_MAX;
    }

    float *d_A, *d_B, *d_C;
    CHECK(cudaMalloc((void**)&d_A, sz));
    CHECK(cudaMalloc((void**)&d_B, sz));
    CHECK(cudaMalloc((void**)&d_C, sz));

    cudaEvent_t ev0, ev1;
    CHECK(cudaEventCreate(&ev0));
    CHECK(cudaEventCreate(&ev1));

    float t_htod = 0.0f, t_kernel = 0.0f, t_dtoh = 0.0f;

    CHECK(cudaEventRecord(ev0));
    CHECK(cudaMemcpy(d_A, h_A, sz, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_B, h_B, sz, cudaMemcpyHostToDevice));
    CHECK(cudaEventRecord(ev1));
    CHECK(cudaEventSynchronize(ev1));
    CHECK(cudaEventElapsedTime(&t_htod, ev0, ev1));

    int blocks = (N + TPB - 1) / TPB;

    CHECK(cudaEventRecord(ev0));
    vec_add<<<blocks, TPB>>>(d_A, d_B, d_C, N);
    CHECK(cudaGetLastError());
    CHECK(cudaEventRecord(ev1));
    CHECK(cudaEventSynchronize(ev1));
    CHECK(cudaEventElapsedTime(&t_kernel, ev0, ev1));

    CHECK(cudaEventRecord(ev0));
    CHECK(cudaMemcpy(h_C, d_C, sz, cudaMemcpyDeviceToHost));
    CHECK(cudaEventRecord(ev1));
    CHECK(cudaEventSynchronize(ev1));
    CHECK(cudaEventElapsedTime(&t_dtoh, ev0, ev1));

    float total = t_htod + t_kernel + t_dtoh;

    bool ok = true;
    for (int i = 0; i < 10; i++) {
        if (fabs(h_C[i] - (h_A[i] + h_B[i])) > 1e-6) {
            ok = false;
            break;
        }
    }

    printf("N=%d TPB=%d BLOCKS=%d HTOD=%.4f KERNEL=%.4f DTOH=%.4f TOTAL=%.4f CORRECT=%d\n",
           N, TPB, blocks, t_htod, t_kernel, t_dtoh, total, (int)ok);

    delete[] h_A;
    delete[] h_B;
    delete[] h_C;

    CHECK(cudaFree(d_A));
    CHECK(cudaFree(d_B));
    CHECK(cudaFree(d_C));
    CHECK(cudaEventDestroy(ev0));
    CHECK(cudaEventDestroy(ev1));

    return 0;
}

Overwriting cuda_files/vector_add_cuda.cu


In [11]:
!nvcc -O2 -std=c++11 -arch=sm_86 cuda_files/vector_add_cuda.cu -o cuda_files/vec_cuda
!./cuda_files/vec_cuda 10000000 256


N=10000000 TPB=256 BLOCKS=39063 HTOD=16.8304 KERNEL=3.3402 DTOH=73.4599 TOTAL=93.6305 CORRECT=1


## Langkah 5: CUDA Pinned Memory – Transfer Lebih Cepat

In [14]:
%%writefile cuda_files/vector_add_pinned.cu
#include <iostream>
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

void checkCuda(cudaError_t err, const char* file, int line) {
    if (err != cudaSuccess) {
        printf("CUDA error %s:%d: %s\n", file, line, cudaGetErrorString(err));
        exit(1);
    }
}

#define CHECK(call) checkCuda((call), __FILE__, __LINE__)

__global__ void vec_add(float* A, float* B, float* C, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        C[idx] = A[idx] + B[idx];
    }
}

int main(int argc, char* argv[]) {
    int N = (argc > 1) ? atoi(argv[1]) : 10000000;
    int TPB = (argc > 2) ? atoi(argv[2]) : 256;
    size_t sz = N * sizeof(float);

    float *h_A, *h_B, *h_C;

    CHECK(cudaHostAlloc((void**)&h_A, sz, cudaHostAllocDefault));
    CHECK(cudaHostAlloc((void**)&h_B, sz, cudaHostAllocDefault));
    CHECK(cudaHostAlloc((void**)&h_C, sz, cudaHostAllocDefault));

    srand(42);
    for (int i = 0; i < N; i++) {
        h_A[i] = (float)rand() / RAND_MAX;
        h_B[i] = (float)rand() / RAND_MAX;
    }

    float *d_A, *d_B, *d_C;
    CHECK(cudaMalloc((void**)&d_A, sz));
    CHECK(cudaMalloc((void**)&d_B, sz));
    CHECK(cudaMalloc((void**)&d_C, sz));

    cudaEvent_t ev0, ev1;
    CHECK(cudaEventCreate(&ev0));
    CHECK(cudaEventCreate(&ev1));

    float t_htod = 0.0f, t_kernel = 0.0f, t_dtoh = 0.0f;

    CHECK(cudaEventRecord(ev0));
    CHECK(cudaMemcpy(d_A, h_A, sz, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_B, h_B, sz, cudaMemcpyHostToDevice));
    CHECK(cudaEventRecord(ev1));
    CHECK(cudaEventSynchronize(ev1));
    CHECK(cudaEventElapsedTime(&t_htod, ev0, ev1));

    int blocks = (N + TPB - 1) / TPB;

    CHECK(cudaEventRecord(ev0));
    vec_add<<<blocks, TPB>>>(d_A, d_B, d_C, N);
    CHECK(cudaGetLastError());
    CHECK(cudaEventRecord(ev1));
    CHECK(cudaEventSynchronize(ev1));
    CHECK(cudaEventElapsedTime(&t_kernel, ev0, ev1));

    CHECK(cudaEventRecord(ev0));
    CHECK(cudaMemcpy(h_C, d_C, sz, cudaMemcpyDeviceToHost));
    CHECK(cudaEventRecord(ev1));
    CHECK(cudaEventSynchronize(ev1));
    CHECK(cudaEventElapsedTime(&t_dtoh, ev0, ev1));

    float total = t_htod + t_kernel + t_dtoh;

    bool ok = true;
    for (int i = 0; i < 10; i++) {
        if (fabs(h_C[i] - (h_A[i] + h_B[i])) > 1e-6) {
            ok = false;
            break;
        }
    }

    printf("N=%d TPB=%d HTOD=%.4f KERNEL=%.4f DTOH=%.4f TOTAL=%.4f CORRECT=%d\n",
           N, TPB, t_htod, t_kernel, t_dtoh, total, (int)ok);

    CHECK(cudaFreeHost(h_A));
    CHECK(cudaFreeHost(h_B));
    CHECK(cudaFreeHost(h_C));

    CHECK(cudaFree(d_A));
    CHECK(cudaFree(d_B));
    CHECK(cudaFree(d_C));

    CHECK(cudaEventDestroy(ev0));
    CHECK(cudaEventDestroy(ev1));

    return 0;
}

Overwriting cuda_files/vector_add_pinned.cu


In [15]:
!nvcc -O2 -std=c++11 -arch=sm_86 cuda_files/vector_add_pinned.cu -o cuda_files/vec_pinned
!./cuda_files/vec_pinned 10000000 256


N=10000000 TPB=256 HTOD=9.7687 KERNEL=3.5765 DTOH=4.9404 TOTAL=18.2856 CORRECT=1


## Langkah 6: Eksperimen Otomatis – Variasi Ukuran Data

In [ ]:
def parse_cpu(out):
    d={}
    for tok in out.split():
        if '=' in tok: k,v=tok.split('='); d[k]=v
    return d

def parse_cuda(out):
    d={}
    for tok in out.split():
        if '=' in tok: k,v=tok.split('='); d[k]=v
    return d

sizes = [1_000, 10_000, 100_000, 1_000_000, 10_000_000, 100_000_000]
results = []

for N in sizes:
    row = {'N': N}
    # CPU Serial
    r = subprocess.run(f'./cuda_files/vec_cpu {N}',   shell=True,capture_output=True,text=True)
    d = parse_cpu(r.stdout); row['cpu_ms']  = float(d.get('TIME_MS',0))
    # OpenMP (4 thread)
    r = subprocess.run(f'./cuda_files/vec_omp {N} 4', shell=True,capture_output=True,text=True)
    d = parse_cpu(r.stdout); row['omp_ms']  = float(d.get('TIME_MS',0))
    # CUDA Basic
    r = subprocess.run(f'./cuda_files/vec_cuda {N} 256',shell=True,capture_output=True,text=True)
    d = parse_cuda(r.stdout)
    row['cuda_htod']   = float(d.get('HTOD',0))
    row['cuda_kernel'] = float(d.get('KERNEL',0))
    row['cuda_dtoh']   = float(d.get('DTOH',0))
    row['cuda_total']  = float(d.get('TOTAL',0))
    # CUDA Pinned
    r = subprocess.run(f'./cuda_files/vec_pinned {N} 256',shell=True,capture_output=True,text=True)
    d = parse_cuda(r.stdout); row['pinned_total'] = float(d.get('TOTAL',0))
    results.append(row)
    print(f'N={N:>12,}: cpu={row["cpu_ms"]:8.2f}ms  omp={row["omp_ms"]:7.2f}ms  cuda={row["cuda_total"]:7.2f}ms  pinned={row["pinned_total"]:7.2f}ms')

df = pd.DataFrame(results)
print('\n=== Eksperimen selesai ===')


N=       1,000: cpu=    0.00ms  omp=   0.57ms  cuda=   7.35ms  pinned=  30.88ms
N=      10,000: cpu=    0.01ms  omp=   0.35ms  cuda=   3.01ms  pinned=   2.76ms
N=     100,000: cpu=    0.06ms  omp=   0.67ms  cuda=   3.78ms  pinned=   4.14ms
N=   1,000,000: cpu=    0.88ms  omp=   0.88ms  cuda=   5.75ms  pinned=   4.25ms
N=  10,000,000: cpu=   12.26ms  omp=   7.62ms  cuda=  67.90ms  pinned=  13.84ms


In [ ]:
# Hitung Speedup dan Bandwidth
df['speedup_omp']    = df['cpu_ms'] / df['omp_ms']
df['speedup_cuda']   = df['cpu_ms'] / df['cuda_total']
df['speedup_pinned'] = df['cpu_ms'] / df['pinned_total']
df['pcie_bw_cuda']   = (2*df['N']*4/1e9) / (df['cuda_htod']+df['cuda_dtoh'])*1000  # GB/s
df['pcie_bw_pinned'] = (2*df['N']*4/1e9) / df['pinned_total']*1000

# Cetak tabel ringkasan
print('=== Tabel Waktu Eksekusi (ms) ===')
print(df[['N','cpu_ms','omp_ms','cuda_total','pinned_total']].to_string(index=False))
print('\n=== Tabel Speedup ===')
print(df[['N','speedup_omp','speedup_cuda','speedup_pinned']].to_string(index=False))


In [ ]:
## Langkah 7: Visualisasi Hasil